# Week 6 — Spark Architecture, File Formats & Data Pipelines
### Dataset: Sample - Superstore.csv (9,994 records)
### Submitted by: Saksham Sharma

In [5]:
!pip install pyspark -q

In [8]:
from google.colab import files
uploaded = files.upload()

Saving Sample_-_Superstore.csv to Sample_-_Superstore.csv


In [24]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr

spark = SparkSession.builder \
    .appName("Week_6_Assignment") \
    .getOrCreate()

# Load data
df = spark.read.csv("Sample_-_Superstore.csv",
                    header=True,
                    inferSchema=True)

# Fix numeric columns (inferSchema sometimes loads as String due to CSV quirks)
df = df.withColumn("Sales",    expr("try_cast(Sales as double)")) \
       .withColumn("Profit",   expr("try_cast(Profit as double)")) \
       .withColumn("Discount", expr("try_cast(Discount as double)")) \
       .withColumn("Quantity", expr("try_cast(Quantity as int)"))

df.printSchema()
df.show(5)
print(f"Total rows: {df.count()}")
print(f"Total columns: {len(df.columns)}")

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----

## Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

**Answer:**

A Spark application consists of three core components that work together to distribute and execute computation across a cluster:

**1. Driver:**
The Driver is the JVM process that runs the main() function of the Spark application. It is responsible for:
- Converting user code into a DAG of stages and tasks
- Negotiating resources with the Cluster Manager
- Scheduling tasks on Executors and tracking their status
- Hosting the SparkContext / SparkSession object

**2. Cluster Manager:**
The Cluster Manager is an external service responsible for allocating compute resources across the cluster. It receives resource requests from the Driver and launches Executor JVMs on worker nodes. Spark supports three built-in cluster managers:
- Standalone (built-in)
- YARN (Hadoop ecosystem)
- Kubernetes (container-based)

The Cluster Manager has no awareness of Spark application logic — it only manages CPU and memory allocation.

**3. Executor:**
Executors are long-lived JVM worker processes launched on cluster nodes. Each Executor:
- Runs Tasks (smallest unit of work) in parallel threads
- Stores intermediate results in memory or disk (caching)
- Reports task progress and results back to the Driver
- Each Spark application gets its own dedicated Executors

**Interaction Flow:**
Driver → requests resources from Cluster Manager → Cluster Manager launches Executors on Worker Nodes → Driver distributes Tasks to Executors → Executors process data and return results to Driver.

## Q2: How does Spark's Lazy Evaluation strategy improve performance when chain-processing large datasets?

**Answer:**

Lazy Evaluation means Spark does NOT execute transformations immediately when they are called. Instead, it records each transformation in a logical plan (DAG) and only triggers actual computation when an Action is called (e.g., .show(), .count(), .write()).

**How it improves performance:**

1. **Whole-Stage Optimization:** Spark's Catalyst optimizer analyzes the full DAG before execution and rewrites it into a more efficient physical plan — reordering filters, eliminating redundant columns, and pushing predicates closer to the data source.

2. **Predicate Pushdown:** Filters are pushed all the way down to the file-scan layer (especially for Parquet), so only matching rows are read from disk — avoiding loading the entire dataset.

3. **Column Pruning:** Only the columns referenced in the query are read from columnar formats, drastically reducing I/O.

4. **Pipelining:** Multiple narrow transformations (map → filter → select) are fused into a single stage and processed partition-by-partition without materializing intermediate DataFrames.

5. **Avoids Unnecessary Work:** If you define 10 transformations but only call .show(5), Spark may only need to process enough data to return 5 rows, short-circuiting the rest.

**Contrast with Pandas:** Pandas executes eagerly — every line runs immediately with no opportunity for cross-operation optimization. Spark's lazy model is what enables query optimization at scale.

## Q3: Write a Spark command to read a CSV file ensuring the first row is treated as a header and inferSchema is enabled.

**Answer:**

Use `spark.read.csv()` with `header=True` so Spark uses row 1 as column names, and `inferSchema=True` so Spark automatically detects column data types by sampling the data.

In [16]:
# Read CSV with header and inferSchema
df = spark.read.csv("Sample_-_Superstore.csv",
                    header=True,
                    inferSchema=True)

# Verify schema and shape
df.printSchema()
print(f"Total rows: {df.count()}")
print(f"Total columns: {len(df.columns)}")
df.show(5)

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)

Total rows: 9994
Total columns: 21
+------+--------------+----------+----------+--------------+-----------+---------------+---------+------------

## Q4: What is the difference between CSV and Parquet in terms of storage and performance?

**Answer:**

| Dimension | CSV (Row-Based) | Parquet (Columnar) |
|---|---|---|
| Storage Layout | All fields of a row stored together | All values of a column stored together |
| Read Pattern | Must read ALL columns even if 1 needed | Reads ONLY requested columns (column pruning) |
| Compression | Poor — mixed types in one block | Excellent — same-type values compress well |
| Schema | No embedded schema — inferred at read time | Schema embedded in file footer |
| Predicate Pushdown | Not supported — full scan always | Supported — min/max stats skip row groups |
| Best For | Small files, human-readable, simple export | Large-scale analytics, ML pipelines |

**Key insight:** If your query is `SELECT Sales, Profit WHERE Region='West'`, Parquet reads only 3 columns out of 21, skipping 86% of the file. CSV reads all 21 columns for every row regardless.

In [17]:
import time

# Write as Parquet
df.write.mode("overwrite").parquet("superstore_parquet/")

# Write as CSV
df.write.mode("overwrite").option("header", "true").csv("superstore_csv_out/")

# Read back both and compare
df_parquet = spark.read.parquet("superstore_parquet/")
df_csv_out = spark.read.csv("superstore_csv_out/", header=True, inferSchema=True)

# Time comparison — Parquet selective read
start = time.time()
df_parquet.filter(df_parquet["Region"] == "West").select("Sales", "Profit").count()
parquet_time = time.time() - start

start = time.time()
df_csv_out.filter(df_csv_out["Region"] == "West").select("Sales", "Profit").count()
csv_time = time.time() - start

print(f"Parquet selective read time: {parquet_time:.3f}s")
print(f"CSV selective read time:     {csv_time:.3f}s")

df_parquet.show(5)

Parquet selective read time: 0.254s
CSV selective read time:     0.308s
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      

## Q5: Write a query to select Product ID and Sales columns where Category is 'Technology'.

**Answer:**

Chain `.filter()` to apply the row predicate, then `.select()` to project only the required columns. This is more efficient than selecting all columns first — Spark's optimizer pushes column pruning to the scan layer.

In [18]:
from pyspark.sql.functions import col

df_technology = df.filter(col("Category") == "Technology") \
                   .select("Product ID", "Sales")

df_technology.show(10)
print(f"Total Technology records: {df_technology.count()}")

+---------------+--------+
|     Product ID|   Sales|
+---------------+--------+
|TEC-PH-10002275| 907.152|
|TEC-PH-10002033| 911.424|
|TEC-PH-10001949|  213.48|
|TEC-AC-10003027|   90.57|
|TEC-PH-10004977|1097.544|
|TEC-PH-10000486| 371.168|
|TEC-PH-10004093| 147.168|
|TEC-AC-10000171|   45.98|
|TEC-AC-10002167|      45|
|TEC-PH-10003988|    21.8|
+---------------+--------+
only showing top 10 rows
Total Technology records: 1847


## Q6: Write code to rename a column and cast the Sales column from String to Double.

**Answer:**

Use `.withColumnRenamed()` to rename a column and `.withColumn()` with `.cast()` to change a column's data type. Both return new DataFrames — the original is unchanged because Spark DataFrames are immutable.

In [19]:
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import col

# Rename Sub-Category to SubCategory
df_renamed = df.withColumnRenamed("Sub-Category", "SubCategory")

# Cast Sales to Double explicitly
df_casted = df_renamed.withColumn("Sales", col("Sales").cast(DoubleType()))

print("Schema after rename and cast:")
df_casted.printSchema()
df_casted.select("SubCategory", "Sales").show(5)

Schema after rename and cast:
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)

+-----------+--------+
|SubCategory|   Sales|
+-----------+--------+
|  Bookcases|  261.96|
|     Chairs|  731.94|
|

## Q7: How does the Lineage Graph (DAG) provide fault tolerance in Spark?

**Answer:**

The Lineage Graph (DAG — Directed Acyclic Graph) is Spark's mechanism for tracking the complete transformation history of every DataFrame. Each node in the DAG is an RDD/DataFrame and each edge is a transformation. This graph is the foundation of Spark's fault tolerance.

**How it works:**

1. **No data replication needed:** Unlike Hadoop HDFS which replicates data 3× for fault tolerance, Spark stores only the transformation recipe (lineage), not the data itself.

2. **Partition-level recovery:** If a worker node fails, Spark identifies EXACTLY which partitions were lost (not the whole dataset). It replays only the subset of the DAG needed to recompute those specific partitions from the last stable checkpoint or source.

3. **Re-computation from source:** Given the DAG: CSV Read → filter → groupBy → agg, if the groupBy stage loses partition 3, Spark re-reads only the relevant source rows, re-applies filter, and re-computes the lost partition.

4. **Checkpointing for deep lineages:** For very long chains (100+ transformations), re-computation from source is expensive. `df.checkpoint()` materializes the DataFrame to disk, breaking the lineage and creating a new recovery point.

**Key difference from database replication:** Spark trades storage space (no data copies) for compute time (re-run transformations). For short DAGs on fast storage (S3/HDFS), re-computation is faster than replication overhead.

## Q8: Write a query to filter rows where status is 'Completed' AND amount is greater than 1000.

**Answer:**

Use `.filter()` with the `&` (AND) operator combining both conditions. In Superstore context: we filter where `Segment == 'Corporate'` (maps to Completed status) AND `Sales > 1000` (maps to amount > 1000).

In [22]:
from pyspark.sql.functions import col, expr

# First cast Sales to Double safely, then filter
df_filtered = df.withColumn("Sales", expr("try_cast(Sales as double)")) \
                .filter(
                    (col("Segment") == "Corporate") &
                    (col("Sales") > 1000)
                )

df_filtered.select("Order ID", "Segment", "Sales", "Category").show(10)
print(f"Records matching filter: {df_filtered.count()}")

+--------------+---------+--------+---------------+
|      Order ID|  Segment|   Sales|       Category|
+--------------+---------+--------+---------------+
|CA-2016-117590|Corporate|1097.544|     Technology|
|CA-2016-105816|Corporate| 1029.95|     Technology|
|CA-2014-106376|Corporate|1113.024|Office Supplies|
|CA-2016-114489|Corporate| 1951.84|      Furniture|
|CA-2015-146262|Corporate|  1188.0|     Technology|
|US-2014-106992|Corporate|3059.982|     Technology|
|US-2014-106992|Corporate|2519.958|     Technology|
|CA-2016-155516|Corporate| 1043.92|      Furniture|
|US-2017-134481|Corporate|1488.424|      Furniture|
|CA-2017-100650|Corporate| 1295.78|Office Supplies|
+--------------+---------+--------+---------------+
only showing top 10 rows
Records matching filter: 153


## Q9: Explain Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

**Answer:**

Predicate Pushdown is a query optimization technique where filter conditions (predicates) are pushed as far down the execution plan as possible — all the way to the file-scan layer — so that only data satisfying the filter is loaded into memory.

**How Parquet enables Predicate Pushdown:**

1. **Row Groups:** Parquet files are divided into Row Groups (default ~128MB each). Each Row Group stores data for a subset of rows.

2. **Column Chunk Statistics:** Each Column Chunk inside a Row Group stores metadata: min value, max value, and null count for that chunk.

3. **Skip at scan time:** When Spark applies `filter(col('Sales') > 5000)`, it checks the statistics — if a Row Group's Sales max = 3000, the entire Row Group is skipped without loading it into memory.

4. **Result:** For a 1TB Parquet file with filter on Sales > 5000 (5% selectivity), Spark may scan only 50GB — a 20× reduction in I/O and memory usage.

**CSV does NOT support Predicate Pushdown** — every row must be read and parsed before filtering can occur. This is one of the biggest performance advantages of Parquet for analytical workloads.

## Q10: Write code to add a new column final_price which is Sales multiplied by 1.18 (18% tax).

**Answer:**

Use `.withColumn()` to derive a new column from an existing one. Spark expressions support arithmetic operators directly on `col()` objects — this is a narrow transformation (no shuffle needed) so it's very fast even on large datasets.

In [27]:
from pyspark.sql.functions import col, round as spark_round

# Add final_price = Sales * 1.18 (with 18% tax)
df_with_tax = df.withColumn(
    "final_price",
    spark_round(col("Sales") * 1.18, 2)
)

df_with_tax.select(
    "Order ID",
    col("Sales").alias("base_price"),
    "final_price",
    "Category"
).show(8)

+--------------+----------+-----------+---------------+
|      Order ID|base_price|final_price|       Category|
+--------------+----------+-----------+---------------+
|CA-2016-152156|    261.96|     309.11|      Furniture|
|CA-2016-152156|    731.94|     863.69|      Furniture|
|CA-2016-138688|     14.62|      17.25|Office Supplies|
|US-2015-108966|  957.5775|    1129.94|      Furniture|
|US-2015-108966|    22.368|      26.39|Office Supplies|
|CA-2014-115812|     48.86|      57.65|      Furniture|
|CA-2014-115812|      7.28|       8.59|Office Supplies|
|CA-2014-115812|   907.152|    1070.44|     Technology|
+--------------+----------+-----------+---------------+
only showing top 8 rows


## Q11: What is the difference between Transformations and Actions? Give two examples of each.

**Answer:**

| Aspect | Transformations | Actions |
|---|---|---|
| Execution | Lazy — only added to DAG | Eager — triggers full DAG execution |
| Return type | New DataFrame / RDD | Value, list, or side-effect (write) |
| Cluster work | No jobs submitted | Submits Spark job to cluster |
| Purpose | Define the transformation logic | Materialize and return results |

**Transformation Examples:**
1. `df.filter(col("Region") == "West")` — selects rows, no job triggered
2. `df.groupBy("Category").agg(avg("Sales"))` — groups data, still no job

**Action Examples:**
1. `df.show()` — prints rows to console, JOB SUBMITTED NOW
2. `df.count()` — returns row count as Python int, JOB SUBMITTED NOW

**Key insight:** You can chain 10 transformations and zero jobs will run. The moment you call one action, Spark optimizes and executes the entire chain at once.

In [28]:
from pyspark.sql.functions import col, avg

# TRANSFORMATIONS (lazy — no execution yet)
df_west = df.filter(col("Region") == "West")        # No job
df_agg = df_west.groupBy("Category") \
                .agg(avg("Sales").alias("avg_sales")) # Still no job

# ACTIONS (trigger execution)
df_agg.show()                    # JOB 1 triggered here
total = df_west.count()          # JOB 2 triggered here
print(f"West region orders: {total}")

+---------------+------------------+
|       Category|         avg_sales|
+---------------+------------------+
|Office Supplies|117.48907552370453|
|      Furniture|360.59540420899896|
|     Technology|422.64417449664415|
+---------------+------------------+

West region orders: 3203


## Q12: Write a code snippet that handles null values and filters the dataset efficiently.

**Answer:**

Always handle nulls BEFORE aggregations to avoid silent result skewing or crashes. Check null counts per column first, then use `.na.fill()` for numeric columns and `.na.drop()` for rows where key columns are missing.

In [29]:
from pyspark.sql.functions import col, count, when

# Step 1: Check null counts across all columns
print("Null counts per column:")
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

# Step 2: Fill nulls in numeric columns with 0
df_clean = df.na.fill({
    "Sales": 0,
    "Profit": 0,
    "Discount": 0,
    "Quantity": 0
})

# Step 3: Drop rows where critical columns are null
df_clean = df_clean.na.drop(subset=["Order ID", "Customer ID"])

# Step 4: Verify
print(f"Rows before cleaning: {df.count()}")
print(f"Rows after cleaning:  {df_clean.count()}")
df_clean.show(5)

Null counts per column:
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|  300|     300|      11|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+---

## Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

### Answer

Spark applications can be deployed in two execution modes: **Client Mode** and **Cluster Mode**. The primary difference lies in where the **Driver Program** executes.

| Client Mode | Cluster Mode |
|--------------|--------------|
| Driver runs on the client machine where the application is submitted. | Driver runs inside the cluster on one of the worker nodes. |
| Client machine must remain connected until the application completes. | Application continues even if the client disconnects. |
| Suitable for development, testing, and debugging. | Suitable for production environments and large-scale jobs. |
| Easier to debug because logs are available locally. | Logs must be accessed from the cluster manager. |
| Failure of the client machine stops the application. | More fault tolerant because the Driver runs inside the cluster. |

### Working

**Client Mode**
1. User submits the application.
2. Driver starts on the local machine.
3. Driver requests resources from the Cluster Manager.
4. Executors are launched on worker nodes.
5. Driver communicates directly with Executors.

**Cluster Mode**
1. User submits the application.
2. Cluster Manager launches the Driver inside the cluster.
3. Driver requests Executors.
4. Executors execute the tasks.
5. Results are returned without depending on the client machine.

### Conclusion

Client Mode is mainly used during development and debugging, whereas Cluster Mode is preferred in production because it provides better reliability, scalability, and fault tolerance.

## Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

### Answer

The given Superstore dataset does not contain the **priority** column and also does not have a **North** region. Therefore, the equivalent columns available in the dataset are used.

- region → Region
- priority → Segment
- North → West
- High → Consumer

The following query demonstrates the required OR condition using the available columns.

In [31]:
from pyspark.sql.functions import col

filtered_df = df.filter(
    (col("Region") == "West") |
    (col("Segment") == "Consumer")
)

filtered_df.show(10)
print(f"Filtered Records: {filtered_df.count()}")

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

## Q15: When exploring a dataset, why is it safer to use `.show(5)` instead of `.collect()` on a multi-terabyte dataset?

### Answer

Both `.show()` and `.collect()` are **Spark Actions**, meaning they trigger the execution of the DAG. However, they behave very differently when working with large datasets.

### `.show(5)`

- Displays only the first five rows of the DataFrame.
- Transfers only a small amount of data from the Executors to the Driver.
- Uses very little Driver memory.
- Ideal for inspecting the structure and sample records of a dataset.

### `.collect()`

- Retrieves **all rows** from every partition.
- Transfers the complete dataset to the Driver process.
- Can consume a huge amount of memory.
- On very large datasets, it may cause **OutOfMemory (OOM)** errors and crash the Driver application.

### Comparison

| `.show(5)` | `.collect()` |
|------------|--------------|
| Displays only a few rows | Returns the complete dataset |
| Low memory usage | High memory usage |
| Safe for large datasets | Unsafe for very large datasets |
| Used for quick inspection | Used only when the entire dataset is genuinely required |

### Conclusion

For large-scale Spark applications, `.show(5)` is the recommended way to preview data because it is memory efficient and avoids transferring the complete dataset to the Driver. The `.collect()` function should only be used on small datasets where bringing all records into memory is safe.

In [33]:
# Save CSV
filtered_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output_csv")

# Save Parquet
filtered_df.write \
    .mode("overwrite") \
    .parquet("output_parquet")

# Convert CSV output to a single CSV file
filtered_df.toPandas().to_csv("filtered_data.csv", index=False)

# Zip the Parquet folder
import shutil
shutil.make_archive("filtered_data_parquet", "zip", "output_parquet")

# Download both files
from google.colab import files

files.download("filtered_data.csv")
files.download("filtered_data_parquet.zip")

print("CSV and Parquet downloaded successfully.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CSV and Parquet downloaded successfully.
